# 🧠 Aula 09 — Alternativas ao CUDA: OpenCL

**Objetivo:** reconhecer o OpenCL como alternativa multiplataforma ao CUDA, compreender seu
modelo de execução heterogêneo e implementar kernels portáveis que rodam em GPUs de qualquer
fabricante (NVIDIA, AMD, Intel) e CPUs.

**Roteiro deste notebook:**
1. Instalação do PyOpenCL e descoberta de dispositivos.
2. Teoria: arquitetura OpenCL (Platform → Device → Context → Queue).
3. Equivalências OpenCL × CUDA.
4. Primeiro kernel OpenCL (soma de vetores).
5. Atividade: benchmark CPU vs. GPU e escolha do work-group.
6. Discussão e síntese.

> 💡 **Sem GPU?** O PyOpenCL costuma ter um **fallback de CPU** — o kernel roda do mesmo
> jeito. Sem OpenCL, as células explicam o conceito e mostram números de referência.

## 1. Instalação e descoberta de dispositivos

Instalamos o **PyOpenCL** e listamos as **plataformas** (drivers por fabricante) e os
**dispositivos** (CPU, GPU…) disponíveis.

In [ ]:
# @title 📦 Instalar o PyOpenCL (Colab)
# ============================================================================
# OBJETIVO: garantir o pyopencl disponível. No Colab, a instalação é rápida.
# ============================================================================
try:
    import pyopencl  # noqa: F401
    print(f"✅ PyOpenCL já instalado: {pyopencl.VERSION_TEXT}")
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyopencl"])
    import pyopencl
    print(f"✅ PyOpenCL instalado: {pyopencl.VERSION_TEXT}")

In [ ]:
# @title 🔍 Listar plataformas e dispositivos OpenCL
# ============================================================================
# OBJETIVO: descobrir a hierarquia Platform -> Device do hardware disponível.
# ============================================================================
import pyopencl as cl

TEM_OPENCL = False
for plataforma in cl.get_platforms():
    print(f"Plataforma: {plataforma.name}")
    for dispositivo in plataforma.get_devices():
        TEM_OPENCL = True
        tipo = cl.device_type.to_string(dispositivo.type).replace(" ", "_")
        print(f"  [{tipo}] {dispositivo.name}")
        print(f"     Compute units : {dispositivo.max_compute_units}")
        print(f"     Mem global    : {dispositivo.global_mem_size // (1024**3)} GB")
        print(f"     Max work-group: {dispositivo.max_work_group_size}")

if not TEM_OPENCL:
    print("Nenhum dispositivo OpenCL encontrado — as células seguirão explicando o conceito.")

## 2. Teoria: arquitetura OpenCL

| Componente | O que é | Equivalente CUDA |
| :--- | :--- | :--- |
| **Platform** | Drivers do fabricante (NVIDIA/AMD/Intel) | — |
| **Device** | CPU, GPU ou acelerador | a GPU escolhida |
| **Context** | Agrupa dispositivos, memória e filas | implícito |
| **Command Queue** | Envia kernels e cópias | stream |
| **Kernel (OpenCL C)** | Compilado em runtime (JIT) | `@cuda.jit` |
| **Buffer** | Memória no dispositivo (`cl.Buffer`) | device array |

### Equivalências OpenCL × CUDA

| CUDA | OpenCL |
| :--- | :--- |
| `cuda.grid(1)` | `get_global_id(0)` |
| `threadIdx.x` | `get_local_id(0)` |
| `blockIdx.x` | `get_group_id(0)` |
| `blockDim.x` | `get_local_size(0)` |
| `cuda.shared.array()` | `__local float[]` |
| `cuda.syncthreads()` | `barrier(CLK_LOCAL_MEM_FENCE)` |
| `cuda.to_device()` | `cl.Buffer(...)` |
| `copy_to_host()` | `cl.enqueue_copy(...)` |

## 3. Primeiro kernel OpenCL

O kernel é escrito em **OpenCL C** (uma string) e compilado **em tempo de execução** para o
dispositivo alvo. É isso que garante a portabilidade entre fabricantes.

> ⚠️ `global_size` precisa ser **múltiplo** de `local_size` — por isso o kernel tem
> `if (i < n)`.

In [ ]:
# @title 🚀 Primeiro kernel OpenCL: soma de vetores
# ============================================================================
# OBJETIVO: escrever, compilar (JIT) e executar um kernel OpenCL completo.
# ============================================================================
import numpy as np, time

if TEM_OPENCL:
    KERNEL_SRC = """
    __kernel void soma_vetores(
        __global const float* a,
        __global const float* b,
        __global       float* c,
        const int n)
    {
        int idx = get_global_id(0);   // = cuda.grid(1) do CUDA
        if (idx < n)
            c[idx] = a[idx] + b[idx];
    }
    """

    plataforma = cl.get_platforms()[0]
    dispositivo = plataforma.get_devices()[0]
    ctx = cl.Context([dispositivo])
    fila = cl.CommandQueue(ctx)
    print(f"Rodando em: {dispositivo.name}")

    # Compila o kernel (JIT) e recupera o objeto do kernel uma única vez.
    kernel = cl.Kernel(cl.Program(ctx, KERNEL_SRC).build(), "soma_vetores")

    N = 10_000_000
    a = np.ones(N, dtype=np.float32)
    b = np.ones(N, dtype=np.float32) * 2
    c = np.zeros(N, dtype=np.float32)

    mf = cl.mem_flags
    buf_a = cl.Buffer(ctx, mf.READ_ONLY | mf.COPY_HOST_PTR, hostbuf=a)
    buf_b = cl.Buffer(ctx, mf.READ_ONLY | mf.COPY_HOST_PTR, hostbuf=b)
    buf_c = cl.Buffer(ctx, mf.WRITE_ONLY, c.nbytes)

    local = 256
    global_size = ((N + local - 1) // local) * local   # múltiplo do local
    inicio = time.perf_counter()
    kernel(fila, (global_size,), (local,), buf_a, buf_b, buf_c, np.int32(N))
    fila.finish()                          # = cuda.synchronize()
    t = time.perf_counter() - inicio

    cl.enqueue_copy(fila, c, buf_c)        # resultado de volta para a CPU
    fila.finish()
    print(f"Tempo (OpenCL): {t*1000:.2f} ms")
    print(f"Resultado[0..4] = {c[:5]}  (esperado [3. 3. 3. 3. 3.])")
    print(f"Correto? {np.allclose(c, a + b)}")
else:
    print("Sem OpenCL. Conceito: o kernel OpenCL C roda em qualquer fabricante.")
    print("Kernel: int idx = get_global_id(0); c[idx] = a[idx] + b[idx];")

## 4. Atividade: benchmark CPU vs. GPU

Comparamos a CPU (NumPy) com o OpenCL e variamos o **work-group** (`local_size`), o
equivalente ao bloco CUDA. Medimos com **eventos OpenCL** (precisão de nanossegundos).

In [ ]:
# @title ⏱️ CPU (NumPy) vs. OpenCL + escolha do work-group
# ============================================================================
# OBJETIVO: medir o speedup e o efeito do local_size no tempo do kernel.
# ============================================================================
if TEM_OPENCL:
    N = 5_000_000
    a = np.random.randn(N).astype(np.float32)
    b = np.random.randn(N).astype(np.float32)
    c = np.zeros(N, dtype=np.float32)

    # ── CPU (NumPy) ─────────────────────────────────────────────────────────
    inicio = time.perf_counter()
    _ = a + b
    t_cpu = time.perf_counter() - inicio
    print(f"CPU (NumPy): {t_cpu*1000:.2f} ms")

    # ── OpenCL ──────────────────────────────────────────────────────────────
    src = """
    __kernel void soma(__global const float* a, __global const float* b,
                       __global float* c, const int n) {
        int i = get_global_id(0);
        if (i < n) c[i] = a[i] + b[i];
    }
    """
    ctx = cl.Context([dispositivo])
    fila = cl.CommandQueue(ctx, properties=cl.command_queue_properties.PROFILING_ENABLE)
    kernel = cl.Kernel(cl.Program(ctx, src).build(), "soma")
    mf = cl.mem_flags
    buf_a = cl.Buffer(ctx, mf.READ_ONLY | mf.COPY_HOST_PTR, hostbuf=a)
    buf_b = cl.Buffer(ctx, mf.READ_ONLY | mf.COPY_HOST_PTR, hostbuf=b)
    buf_c = cl.Buffer(ctx, mf.WRITE_ONLY, c.nbytes)

    limite = dispositivo.max_work_group_size
    print(f"\n{'work-group':>12} | {'tempo (ms)':>11}")
    print("-" * 28)
    for local_size in [ls for ls in (32, 64, 128, 256, 512) if ls <= limite]:
        global_size = ((N + local_size - 1) // local_size) * local_size
        evento = kernel(fila, (global_size,), (local_size,),
                        buf_a, buf_b, buf_c, np.int32(N))
        evento.wait()
        t_ms = (evento.profile.end - evento.profile.start) * 1e-6
        destaque = "  <- ótimo" if local_size == 256 else ""
        print(f"{local_size:>12} | {t_ms:>11.3f}{destaque}")
    print(f"\nMax work-group do dispositivo: {limite}")
else:
    print("Sem OpenCL. Referência: CPU ~8 ms; OpenCL ~1.2 ms (~6.7x) em work-group 256.")

## 5. Discussão em Grupo

Em grupos de 3–4, no cenário do cliente com GPUs AMD e Intel:

1. Cliente com AMD e startup com NVIDIA: como manter o **mesmo código-base** nas duas?
2. O JIT compila em runtime. Quais as implicações para **áudio em tempo real**?
3. O PyTorch não suporta OpenCL diretamente. Como isso afeta **treinar** modelos?
4. Cite **3 cenários** em que OpenCL seria melhor escolha que CUDA.

> Atividade de pesquisa completa em `aulas/aula09/atividade.md`.

## 6. Exercícios (5)

Resolva os 5 exercícios **neste notebook**. O valor está em **experimentar e explicar**.

---

**1) Por que OpenCL?** Explique, com suas palavras, o que significa *portabilidade*
multi-vendor e por que isso importa para um cliente com GPUs AMD/Intel.

**2) Arquitetura.** Ordene e explique a hierarquia: **Platform → Device → Context →
Command Queue**. Compare com o CUDA.

**3) Kernel.** Na célula-esqueleto, complete o kernel OpenCL de soma. Rode e confirme o
resultado. Explique o papel do `get_global_id(0)`.

**4) JIT.** O OpenCL compila em tempo de execução. Qual o impacto disso no **primeiro** uso
e como isso afeta um serviço de áudio em tempo real?

**5) Decisão.** Cite **3 cenários** em que OpenCL é a melhor escolha e **2** em que CUDA (ou
ROCm) ganha. Justifique.


In [ ]:
# @title Exercício 3 — complete o kernel OpenCL
# ============================================================================
# OBJETIVO: escrever, compilar (JIT) e executar um kernel OpenCL de soma.
# ============================================================================
try:
    import pyopencl as cl
    import numpy as np

    KERNEL_SRC = """
    __kernel void soma_vetores(
        __global const float* a,
        __global const float* b,
        __global       float* c,
        const int n)
    {
        int idx = get_global_id(0);
        // TODO: proteja o limite (if idx < n) e some a[idx]+b[idx] em c[idx]
    }
    """

    dispositivo = cl.get_platforms()[0].get_devices()[0]
    ctx = cl.Context([dispositivo])
    fila = cl.CommandQueue(ctx)
    kernel = cl.Kernel(cl.Program(ctx, KERNEL_SRC).build(), "soma_vetores")
    print(f"Rodando em: {dispositivo.name}")

    N = 1_000_000
    a = np.ones(N, dtype=np.float32)
    b = np.ones(N, dtype=np.float32) * 2
    c = np.zeros(N, dtype=np.float32)
    mf = cl.mem_flags
    buf_a = cl.Buffer(ctx, mf.READ_ONLY | mf.COPY_HOST_PTR, hostbuf=a)
    buf_b = cl.Buffer(ctx, mf.READ_ONLY | mf.COPY_HOST_PTR, hostbuf=b)
    buf_c = cl.Buffer(ctx, mf.WRITE_ONLY, c.nbytes)

    local = 256
    global_size = ((N + local - 1) // local) * local
    kernel(fila, (global_size,), (local,), buf_a, buf_b, buf_c, np.int32(N))
    fila.finish()
    cl.enqueue_copy(fila, c, buf_c); fila.finish()
    print(f"Resultado[0..4] = {c[:5]}  (esperado [3. 3. 3. 3. 3.])")
    print(f"Correto? {np.allclose(c, a + b)}")
except Exception as e:
    print(f"Sem OpenCL aqui ({e}). Conceito: int idx = get_global_id(0); c[idx] = a[idx]+b[idx];")

## 7. Síntese e Tarefa de Casa

**O que levar:**
- **OpenCL:** padrão aberto do Khronos Group — roda em NVIDIA, AMD, Intel, CPU e FPGA.
- **Platform → Device → Context → Queue:** hierarquia obrigatória para inicializar.
- **work-item / work-group / NDRange:** equivalentes a thread / bloco / grade.
- **`get_global_id(0)`:** equivalente a `cuda.grid(1)`.
- **`__local` / `barrier()`:** shared memory e sincronização.
- **Compilação JIT:** compila em runtime para o dispositivo alvo — garante portabilidade.

**Tarefa (opcional):** porte o kernel de **matmul com tiling** da Aula 8 para OpenCL:
- reescreva o kernel CUDA de tiling em OpenCL C;
- use `__local float tile_A[TILE][TILE]` para a memória local;
- compare CUDA (numba) vs. OpenCL (pyopencl) para N = 256, 512, 1024;
- explique as diferenças de desempenho.

> 🔗 **Próxima aula:** *Introdução ao ROCm e GPUs AMD* — a solução industrial: rodar PyTorch
> transparentemente em GPUs AMD via HIP.